In [ ]:
#
# For licensing see accompanying LICENSE file.
# Copyright (C) 2025 Apple Inc. All Rights Reserved.
#

In [ ]:
import os
import hydra
import torch
from einops import rearrange, repeat
import omegaconf
from omegaconf import DictConfig, OmegaConf
from utils.modelrunner import ModelRunner
from torchvision.utils import save_image

In [ ]:
# define the sampler
from model.sampler import ODESampler

sampler = ODESampler(
    num_timesteps=10,
    cfg_scale=1.0,
    t_eps=1e-4,
    sample_step="euler",
    eval_ema=True,
)

In [ ]:
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")

cfg = OmegaConf.load("configs/model/architecture/pixeldit_demo.yaml")
model = hydra.utils.instantiate(cfg)
print(model)

from torch.optim.swa_utils import AveragedModel
model_ema = AveragedModel(
    model,
    multi_avg_fn=torch.optim.swa_utils.get_ema_multi_avg_fn(
        0.999
    ),
    use_buffers=True,
)

# Load the model checkpoint
# model.load_state_dict(
#     torch.load(CKPT_PATH, map_location="cpu")["state_dict"],
#     strict=False,
# )
# model_ema.load_state_dict(
#     torch.load(EMA_CKPT_PATH, map_location="cpu")["state_dict"],
#     strict=False,
# )

model = model.to(device)
model_ema = model_ema.to(device)
model.eval()
model_ema.eval()

In [ ]:
from model.path import LinearPath

path = LinearPath()

sampler.setup(
    model=model,
    model_ema=model_ema,
    path=path,
)

In [ ]:

num_samples = 1
label = torch.randint(0, 1000, (num_samples,), device=device)
res_lo = 256 # Low resolution which the model is trained on
res_hi = 512 # High resolution which we want to sample at

# we keep the context the same point as the low resolution in training
# so that we can use whatever resolution we want to sample as queries

y_noise_lo = torch.randn(
    num_samples, res_lo ** 2, 3, device=device
)
y_noise_hi = torch.randn(
    num_samples, res_hi ** 2, 3, device=device
)
y_noise = torch.cat([y_noise_lo, y_noise_hi], dim=1)

x_channel = torch.linspace(0, 1, 256).view(1, 1, -1).repeat(1, 256, 1)
y_channel = torch.linspace(0, 1, 256).view(1, -1, 1).repeat(1, 1, 256)
x_lo = torch.cat((x_channel, y_channel), dim=0)
x_lo = repeat(x_lo, 'c h w -> b (h w) c', b=num_samples).to(device)
x_channel = torch.linspace(0, 1, res_hi).view(1, 1, -1).repeat(1, res_hi, 1)
y_channel = torch.linspace(0, 1, res_hi).view(1, -1, 1).repeat(1, 1, res_hi)
x_hi = torch.cat((x_channel, y_channel), dim=0)
x_hi = repeat(x_hi, 'c h w -> b (h w) c', b=num_samples).to(device)
x = torch.cat((x_lo, x_hi), dim=1)
print(f"x: {x.shape}, y_noise: {y_noise.shape}")

context_mask = torch.zeros((res_lo ** 2 + res_hi ** 2, ), device=device, dtype=torch.bool)
context_mask[:res_lo ** 2] = True  # Use low-res as context

y_sampled = sampler.sample(
    query_x=x,
    query_y_sampled=y_noise,
    label=label,
    context_mask=context_mask,
)
print(f"y_sampled: {y_sampled.shape}")

y_sampled = (y_sampled + 1) * 0.5
y_sampled_lo = y_sampled[:, :res_lo ** 2]
y_sampled_lo = rearrange(y_sampled_lo, "b (h w) c -> b c h w", h=res_lo)
y_sampled_hi = y_sampled[:, res_lo ** 2:]
y_sampled_hi = rearrange(y_sampled_hi, "b (h w) c -> b c h w", h=res_hi)

save_image(y_sampled_lo, 'img_low.png')
save_image(y_sampled_hi, 'img_high.png')